In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, LongType
)
from datetime import datetime

# ── Path configuration ──────────────────────────────────────────────────────
RAW       = "/Volumes/mini_project_team_a3t7/default/mini_project/raw"
BRONZE    = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze"
CKPT_BASE = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze"

# Convenience sub-paths
def raw(f):    return f"{RAW}/{f}"
def bronze(f): return f"{BRONZE}/{f}"
def ckpt(f):   return f"{CKPT_BASE}/{f}"

# Standard metadata columns added to every Bronze table
INGESTION_TS = F.current_timestamp().alias("_ingestion_timestamp")
BATCH_DATE   = F.current_date().alias("_batch_date")

print("Paths configured.")
print(f"  RAW    : {RAW}")
print(f"  BRONZE : {BRONZE}")
print(f"  CKPT   : {CKPT_BASE}")

In [0]:
pos_stream_schema = StructType([
    StructField("transaction_id",  StringType(),    False),
    StructField("store_id",        StringType(),    True),
    StructField("product_id",      StringType(),    True),
    StructField("customer_id",     StringType(),    True),
    StructField("timestamp",       StringType(),    True),  # kept as string; cast in Silver
    StructField("quantity",        IntegerType(),   True),
    StructField("unit_price",      DoubleType(),    True),
    StructField("total_amount",    DoubleType(),    True),
    StructField("payment_method",  StringType(),    True),
    StructField("channel",         StringType(),    True),
])

# FIX 1: spark.readStream instead of spark.read — enables streaming ingestion
df_pos_stream = (
    spark.readStream                                      # FIXED: was spark.read
         .schema(pos_stream_schema)
         .option("maxFilesPerTrigger", 10)               # process 10 files per micro-batch
         .json("/Volumes/mini_project_team_a3t7/default/mini_project/raw/pos_transactions_stream/")
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("pos_stream_jsonl"))
)



In [0]:
pos_stream_query = (
    df_pos_stream.writeStream                            
        .format("delta")       
        .outputMode("append")
        .option("checkpointLocation", ckpt("pos_transactions_stream"))  
        .trigger(availableNow=True)             
        .partitionBy("_batch_date")
        .start("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/pos_transactions_stream/")
)

print("Streaming write started: bronze/pos_transactions_stream")